# Diabetes Prediction - Feature Engineering Demo

**Competition:** Playground Series S5E12  
**Goal:** Predict probability of diabetes diagnosis  
**Evaluation:** ROC AUC

## 1. Import Libraries

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import roc_auc_score

## 2. Load Data

In [3]:
TRAIN_PATH = "./data/train.csv"
TEST_PATH = "./data/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

## 3. Exploratory Data Analysis

In [4]:
train.head()

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
0,0,31,1,45,7.7,6.8,6.1,33.4,0.93,112,...,Female,Hispanic,Highschool,Lower-Middle,Current,Employed,0,0,0,1.0
1,1,50,2,73,5.7,6.5,5.8,23.8,0.83,120,...,Female,White,Highschool,Upper-Middle,Never,Employed,0,0,0,1.0
2,2,32,3,158,8.5,7.4,9.1,24.1,0.83,95,...,Male,Hispanic,Highschool,Lower-Middle,Never,Retired,0,0,0,0.0
3,3,54,3,77,4.6,7.0,9.2,26.6,0.83,121,...,Female,White,Highschool,Lower-Middle,Current,Employed,0,1,0,1.0
4,4,54,1,55,5.7,6.2,5.1,28.8,0.90,108,...,Male,White,Highschool,Upper-Middle,Never,Retired,0,1,0,1.0


In [5]:
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"\nTarget distribution:")
print(train['diagnosed_diabetes'].value_counts(normalize=True))

Train shape: (700000, 26)
Test shape: (300000, 25)

Target distribution:
diagnosed_diabetes
1.0    0.623296
0.0    0.376704
Name: proportion, dtype: float64


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700000 entries, 0 to 699999
Data columns (total 26 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   id                                  700000 non-null  int64  
 1   age                                 700000 non-null  int64  
 2   alcohol_consumption_per_week        700000 non-null  int64  
 3   physical_activity_minutes_per_week  700000 non-null  int64  
 4   diet_score                          700000 non-null  float64
 5   sleep_hours_per_day                 700000 non-null  float64
 6   screen_time_hours_per_day           700000 non-null  float64
 7   bmi                                 700000 non-null  float64
 8   waist_to_hip_ratio                  700000 non-null  float64
 9   systolic_bp                         700000 non-null  int64  
 10  diastolic_bp                        700000 non-null  int64  
 11  heart_rate                

In [7]:
train.describe()

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,diastolic_bp,heart_rate,cholesterol_total,hdl_cholesterol,ldl_cholesterol,triglycerides,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
count,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000,700000.000000
mean,349999.500000,50.359734,2.072411,80.230803,5.963695,7.002200,6.012733,25.874684,0.858766,116.294193,75.440924,70.167749,186.818801,53.823214,102.905854,123.081850,0.149401,0.181990,0.030324,0.623296
std,202072.738554,11.655520,1.048189,51.195071,1.463336,0.901907,2.022707,2.860705,0.037980,11.010390,6.825775,6.938722,16.730832,8.266545,19.022416,24.739397,0.356484,0.385837,0.171478,0.484560
min,0.000000,19.000000,1.000000,1.000000,0.100000,3.100000,0.600000,15.100000,0.680000,91.000000,51.000000,42.000000,117.000000,21.000000,51.000000,31.000000,0.000000,0.000000,0.000000,0.000000
25%,174999.750000,42.000000,1.000000,49.000000,5.000000,6.400000,4.600000,23.900000,0.830000,108.000000,71.000000,65.000000,175.000000,48.000000,89.000000,106.000000,0.000000,0.000000,0.000000,0.000000
50%,349999.500000,50.000000,2.000000,71.000000,6.000000,7.000000,6.000000,25.900000,0.860000,116.000000,75.000000,70.000000,187.000000,54.000000,103.000000,123.000000,0.000000,0.000000,0.000000,1.000000
75%,524999.250000,58.000000,3.000000,96.000000,7.000000,7.600000,7.400000,27.800000,0.880000,124.000000,80.000000,75.000000,199.000000,59.000000,116.000000,139.000000,0.000000,0.000000,0.000000,1.000000
max,699999.000000,89.000000,9.000000,747.000000,9.900000,9.900000,16.500000,38.400000,1.050000,163.000000,104.000000,101.000000,289.000000,90.000000,205.000000,290.000000,1.000000,1.000000,1.000000,1.000000


In [8]:
train.isnull().sum()

id                                    0
age                                   0
alcohol_consumption_per_week          0
physical_activity_minutes_per_week    0
diet_score                            0
sleep_hours_per_day                   0
screen_time_hours_per_day             0
bmi                                   0
waist_to_hip_ratio                    0
systolic_bp                           0
diastolic_bp                          0
heart_rate                            0
cholesterol_total                     0
hdl_cholesterol                       0
ldl_cholesterol                       0
triglycerides                         0
gender                                0
ethnicity                             0
education_level                       0
income_level                          0
smoking_status                        0
employment_status                     0
family_history_diabetes               0
hypertension_history                  0
cardiovascular_history                0


## 4. Feature Engineering

We'll demonstrate the 4 main areas:
1. Feature Creation
2. Feature Encoding
3. Feature Selection
4. Feature Extraction

In [9]:
def engineer_features(df, is_train=True):
    df = df.copy()
    
    # --- FEATURE CREATION ---
    
    # Health risk score (combination of key metrics)
    df['health_risk_score'] = (df['bmi'] * 0.3 + 
                               df['systolic_bp'] * 0.2 + 
                               df['cholesterol_total'] * 0.15 +
                               df['waist_to_hip_ratio'] * 50)
    
    # Cholesterol ratio (HDL to total)
    df['cholesterol_ratio'] = df['hdl_cholesterol'] / df['cholesterol_total']
    
    # Lifestyle score
    df['lifestyle_score'] = (df['physical_activity_minutes_per_week'] / 50 + 
                             df['diet_score'] * 2 - 
                             df['screen_time_hours_per_day'] * 3 - 
                             df['alcohol_consumption_per_week'] * 2)
    
    # Activity to BMI ratio
    df['activity_per_bmi'] = df['physical_activity_minutes_per_week'] / (df['bmi'] + 1)
    
    # Blood pressure product
    df['bp_product'] = df['systolic_bp'] * df['diastolic_bp']
    
    # Age groups (binning)
    df['age_group'] = pd.cut(df['age'], bins=[0, 35, 50, 65, 100], 
                             labels=['young', 'middle', 'senior', 'elderly'])
    
    return df

train = engineer_features(train, is_train=True)
test = engineer_features(test, is_train=True)

### Feature Encoding

In [10]:
# Identify categorical columns
cat_cols = ['gender', 'ethnicity', 'education_level', 'income_level', 
            'smoking_status', 'employment_status', 'age_group']

# Label encoding for categorical features
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])
    le_dict[col] = le

print("Encoded categorical features")

Encoded categorical features


### Prepare for Modeling

In [11]:
# Separate features and target
X = train.drop(['id', 'diagnosed_diabetes'], axis=1)
y = train['diagnosed_diabetes']
X_test_final = test.drop(['id'], axis=1)

print(f"Feature matrix shape: {X.shape}")
print(f"Number of features: {X.shape[1]}")

Feature matrix shape: (700000, 30)
Number of features: 30


### Feature Selection

In [12]:
# Use SelectKBest to identify top features
selector = SelectKBest(score_func=f_classif, k=20)
X_selected = selector.fit_transform(X, y)
X_test_selected = selector.transform(X_test_final)

# Get selected feature names
selected_features = X.columns[selector.get_support()].tolist()
print(f"Selected {len(selected_features)} features:")
print(selected_features)

Selected 20 features:
['age', 'physical_activity_minutes_per_week', 'diet_score', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history', 'health_risk_score', 'cholesterol_ratio', 'lifestyle_score', 'activity_per_bmi', 'bp_product', 'age_group']


### Feature Extraction (PCA)

In [13]:
# Scale features before PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)
X_test_scaled = scaler.transform(X_test_selected)

# Apply PCA to reduce dimensionality
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Explained variance ratio: {pca.explained_variance_ratio_.sum():.4f}")
print(f"Reduced to {X_pca.shape[1]} components")

Explained variance ratio: 0.8524
Reduced to 10 components


## 5. Model Training

In [14]:
# Split for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

### Logistic Regression

In [15]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, y_pred)
print(f"Validation AUC: {auc:.4f}")

Validation AUC: 0.6946


## 6. Final Predictions and Submission

In [16]:
# Retrain on full dataset
model.fit(X_scaled, y)

# Generate predictions
y_pred_final = model.predict_proba(X_test_scaled)[:, 1]

print("Sample predictions:")
print(y_pred_final[:10])

Sample predictions:
[0.56607507 0.53982346 0.66787343 0.60279559 0.76950639 0.55436588
 0.77894838 0.89527303 0.56447436 0.69802571]


In [17]:
submission = pd.DataFrame({
    'id': test['id'],
    'diagnosed_diabetes': y_pred_final
})

submission.to_csv('submission.csv', index=False)
print("Submission file created!")
submission.head()

Submission file created!


,id,diagnosed_diabetes
0,700000,0.566075
1,700001,0.539823
2,700002,0.667873
3,700003,0.602796
4,700004,0.769506


In [18]:
# Generate submission file
submission = pd.DataFrame({
    'id': test['id'],
    'diagnosed_diabetes': y_pred_final
})

# Save to CSV
submission.to_csv('submission.csv', index=False)

print("Submission file created successfully!")
print(f"\nFirst few rows of submission:")
print(submission.head(10))
print(f"\nShape: {submission.shape}")
print(f"Prediction range: [{submission['diagnosed_diabetes'].min():.4f}, {submission['diagnosed_diabetes'].max():.4f}]")

Submission file created successfully!

First few rows of submission:
       id  diagnosed_diabetes
0  700000            0.566075
1  700001            0.539823
2  700002            0.667873
3  700003            0.602796
4  700004            0.769506
5  700005            0.554366
6  700006            0.778948
7  700007            0.895273
8  700008            0.564474
9  700009            0.698026

Shape: (300000, 2)
Prediction range: [0.0187, 0.9828]


## Next Steps

Try improving the model by:
- Creating more domain-specific features
- Testing different encoding methods (target encoding)
- Using advanced models (LightGBM, XGBoost, CatBoost)
- Tuning hyperparameters
- Ensemble methods